# Lab 2: Seq2Seq Neural Machine Translation
This notebook is the student skeleton for implementing a GRU-based Seq2Seq model.

In [18]:
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cpu


## 1. Load parallel data

In [19]:
def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_src = load_lines("data/train.en")
train_tgt = load_lines("data/train.vi")
dev_src = load_lines("data/dev.en")
dev_tgt = load_lines("data/dev.vi")

print(train_src[:3])
print(train_tgt[:3])


['i am a student', 'i am a teacher', 'he likes football']
['tôi là sinh viên', 'tôi là giáo viên', 'anh ấy thích bóng đá']


## 2. Build vocabularies
Reuse or complete the utilities from Lab 1.

In [20]:
SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

class Vocab:
    def __init__(self):
        self.word2id = {}
        self.id2word = {}
        for token in SPECIAL_TOKENS:
            self.add_word(token)

    def add_word(self, word):
        if word not in self.word2id:
            idx = len(self.word2id)
            self.word2id[word] = idx
            self.id2word[idx] = word

    def build(self, sentences):
        """Build vocabulary from tokenized sentences"""
        for tokens in sentences:
            for token in tokens:
                self.add_word(token)

    def encode(self, sentence):
        """Return list of ids with <BOS> and <EOS>"""
        tokens = tokenize(sentence)
        ids = [self.word2id["<BOS>"]]
        ids.extend(self.word2id.get(token, self.word2id["<UNK>"]) for token in tokens)
        ids.append(self.word2id["<EOS>"])
        return ids

    def decode(self, ids):
        words = []
        for idx in ids:
            token = self.id2word.get(int(idx), "<UNK>")
            if token == "<EOS>":
                break
            if token not in ["<PAD>", "<BOS>"]:
                words.append(token)
        return " ".join(words)
    
    def __len__(self):
        return len(self.word2id)

def tokenize(sentence):
    return sentence.lower().split()

## 3. Encode data and pad mini-batches

In [21]:
def pad_sequences(sequences, pad_id):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padded.append(seq + [pad_id] * (max_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)

# Build vocabularies
src_vocab = Vocab()
tgt_vocab = Vocab()

# Tokenize sentences
tokenized_train_src = [tokenize(s) for s in train_src]
tokenized_train_tgt = [tokenize(s) for s in train_tgt]
tokenized_dev_src = [tokenize(s) for s in dev_src]
tokenized_dev_tgt = [tokenize(s) for s in dev_tgt]

# Build vocabularies
src_vocab.build(tokenized_train_src)
tgt_vocab.build(tokenized_train_tgt)

print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")

# Encode training data
train_src_ids = [src_vocab.encode(s) for s in train_src]
train_tgt_ids = [tgt_vocab.encode(s) for s in train_tgt]
dev_src_ids = [src_vocab.encode(s) for s in dev_src]
dev_tgt_ids = [tgt_vocab.encode(s) for s in dev_tgt]

print(f"Sample source encoding: {train_src_ids[0]}")
print(f"Sample target encoding: {train_tgt_ids[0]}")

Source vocab size: 54
Target vocab size: 59
Sample source encoding: [1, 4, 5, 6, 7, 2]
Sample target encoding: [1, 4, 5, 6, 7, 2]


## 4. Implement Encoder

In [22]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        outputs, hidden = self.rnn(emb)
        return outputs, hidden

## 5. Implement Decoder

In [23]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token, hidden):
        emb = self.embedding(token)
        output, hidden = self.rnn(emb, hidden)
        logits = self.fc(output)
        return logits, hidden

## 6. Implement Seq2Seq with teacher forcing

In [24]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, bos_id, eos_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.bos_id = bos_id
        self.eos_id = eos_id

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.size()
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)

        _, hidden = self.encoder(src)
        decoder_input = tgt[:, 0].unsqueeze(1)

        for t in range(1, tgt_len):
            logits, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t:t+1, :] = logits

            top1 = logits.argmax(dim=-1)
            use_teacher = random.random() < teacher_forcing_ratio
            decoder_input = tgt[:, t].unsqueeze(1) if use_teacher else top1

        return outputs

    @torch.no_grad()
    def greedy_decode(self, src, max_len=20):
        batch_size = src.size(0)
        _, hidden = self.encoder(src)
        decoder_input = torch.full((batch_size, 1), self.bos_id, dtype=torch.long, device=src.device)

        predictions = [[self.bos_id] for _ in range(batch_size)]

        for _ in range(max_len):
            logits, hidden = self.decoder(decoder_input, hidden)
            next_token = logits.argmax(dim=-1)
            decoder_input = next_token

            for i in range(batch_size):
                token_id = int(next_token[i, 0].item())
                predictions[i].append(token_id)
                if token_id == self.eos_id:
                    predictions[i] = predictions[i]

        return predictions

## 7. Train the model

In [25]:
# Suggested hyperparameters
EMB_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 8

# Create model
encoder = Encoder(len(src_vocab), EMB_DIM, HIDDEN_DIM, pad_id=0)
decoder = Decoder(len(tgt_vocab), EMB_DIM, HIDDEN_DIM, pad_id=0)
model = Seq2Seq(encoder, decoder, bos_id=tgt_vocab.word2id["<BOS>"], eos_id=tgt_vocab.word2id["<EOS>"])
model = model.to(DEVICE)

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <PAD> tokens

# Pad training data
train_src_padded = pad_sequences(train_src_ids, pad_id=0).to(DEVICE)
train_tgt_padded = pad_sequences(train_tgt_ids, pad_id=0).to(DEVICE)

# Training loop
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    # Minibatch training
    for i in range(0, len(train_src_padded), BATCH_SIZE):
        src_batch = train_src_padded[i:i+BATCH_SIZE]
        tgt_batch = train_tgt_padded[i:i+BATCH_SIZE]
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(src_batch, tgt_batch, teacher_forcing_ratio=0.5)
        
        # Compute loss (excluding first token <BOS>)
        loss = criterion(logits[:, 1:, :].reshape(-1, len(tgt_vocab)), tgt_batch[:, 1:].reshape(-1))
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / (len(train_src_padded) // BATCH_SIZE)
        print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {avg_loss:.4f}")

Epoch 10/100, Loss: 4.6195
Epoch 20/100, Loss: 3.0491
Epoch 30/100, Loss: 1.8783
Epoch 40/100, Loss: 0.9507
Epoch 50/100, Loss: 0.6894
Epoch 60/100, Loss: 0.3836
Epoch 70/100, Loss: 0.2475
Epoch 80/100, Loss: 0.1603
Epoch 90/100, Loss: 0.1172
Epoch 100/100, Loss: 0.0899


## 8. Evaluate on the dev set

In [26]:
# Evaluation on dev set
model.eval()
dev_src_padded = pad_sequences(dev_src_ids, pad_id=0).to(DEVICE)

predictions = model.greedy_decode(dev_src_padded, max_len=20)

for i in range(min(5, len(dev_src))):
    src_text = dev_src[i]
    ref_text = dev_tgt[i]
    pred_text = tgt_vocab.decode(predictions[i])
    
    print(f"\nSource:      {src_text}")
    print(f"Reference:   {ref_text}")
    print(f"Prediction:  {pred_text}")


Source:      i am a teacher
Reference:   tôi là giáo viên
Prediction:  tôi là giáo viên

Source:      she likes music
Reference:   cô ấy thích âm nhạc
Prediction:  cô ấy thích âm nhạc

Source:      this is a book
Reference:   đây là một cuốn sách
Prediction:  đây là một cuốn sách

Source:      we study machine learning
Reference:   chúng tôi học máy học
Prediction:  chúng tôi học máy học


## 9. Conceptual questions

**1. What is teacher forcing?**

Teacher forcing là một kỹ thuật huấn luyện trong đó decoder nhận được token đúng từ dữ liệu huấn luyện làm đầu vào tại mỗi bước thời gian, thay vì sử dụng dự đoán của chính nó từ bước trước. Điều này tăng tốc độ hội tụ và ổn định quá trình huấn luyện. Tuy nhiên, nó tạo ra exposure bias: tại thời điểm suy diễn, mô hình sử dụng dự đoán của chính nó có thể sai, nhưng trong quá trình huấn luyện nó luôn nhận được đầu vào đúng. Sự không trùng khớp này có thể dẫn đến tích tụ lỗi.

**2. Why can a basic Seq2Seq model struggle with long sentences?**

Mô hình Seq2Seq cơ bản gặp khó khăn với các câu dài do các lý do:
- Bottleneck thông tin: Toàn bộ câu nguồn được nén vào một trạng thái ẩn có kích thước cố định. Các câu dài mất nhiều thông tin hơn.
- Vanishing gradients: Trong quá trình lan truyền ngược qua các chuỗi dài, gradient trở nên rất nhỏ, làm khó khăn để học các phụ thuộc dài hạn.
- Tích tụ lỗi: Những lỗi trong các bước giải mã sớm truyền đến các bước sau.

Các giải pháp bao gồm cơ chế attention (Lab 3) và Transformers (Lab 4).

**3. What distribution does the decoder model at time step t?**

Tại bước thời gian t, decoder mô hình hóa phân bố xác suất có điều kiện: $P(y_t | y_{<t}, x)$ trong đó:
- $y_t$ là token đích tại bước t
- $y_{<t} = (y_1, y_2, \ldots, y_{t-1})$ là các token đích trước đó
- $x$ là câu nguồn

Decoder tính toán logits trên toàn bộ từ vựng và áp dụng softmax để thu được phân bố xác suất trên tất cả các token đích có thể.